# 1. Environment Setup and Imports
Configure the notebook environment, import dependencies, and load the custom `PageRankEngine` module.

In [1]:
import sys
import os
import numpy as np

# Add the parent directory to the system path so we can import our custom engine
sys.path.append(os.path.abspath('..'))
print("Current working directory:", os.getcwd())
from src.pagerank import PageRankEngine


Current working directory: c:\Users\liang\Downloads\Cloud Computing\notebooks


# 2. PageRank Verification: Iterative vs Analytical
Run both PageRank solvers on the Google 10k graph and validate numerical equivalence with an L1 error check.

In [ ]:
# 1. Initialize the Engine with standard Google teleportation probability (p=0.15)
pr_engine = PageRankEngine(teleport_prob=0.15)

# 2. Load the 10k dataset
data_path = '../data/raw/web-Google_10k.zip'
pr_engine.load_from_file(data_path)

# 3. Compute using the Iterative Power Method (Scalable)
print("\n--- Running Iterative Method ---")
R_iterative = pr_engine.solve_iterative(max_iter=150, tol=1e-8)

# 4. Compute using the Analytical Matrix Inversion (Exact)
print("\n--- Running Analytical Method ---")
R_analytical = pr_engine.solve_analytical()

# 5. Display and Compare Results
print("\n--- Top 5 Nodes (Iterative) ---")
for node, score in pr_engine.get_top_k(R_iterative, k=5):
    print(f"Node {node}: {score:.8f}")

print("\n--- Top 5 Nodes (Analytical) ---")
for node, score in pr_engine.get_top_k(R_analytical, k=5):
    print(f"Node {node}: {score:.8f}")

# 6. The Mathematical Proof of Equivalence
# Using L1 norm to find the absolute difference between the two entire vectors
difference = np.linalg.norm(R_iterative - R_analytical, ord=1)
print(f"\nTotal L1 Error between Iterative and Analytical vectors: {difference:.10e}")
if difference < 1e-5:
    print("SUCCESS: The scalable iterative algorithm mathematically matches the closed-form derivation!")
else:
    print("WARNING: Divergence detected.")

Loading graph from ../data/raw/web-Google_10k.zip...
Reading directly from web-Google_10k.txt inside the archive...
Extracting nodes and building index mappings...
Processing 10000 unique nodes and 78323 edges...
Sparse transition matrix M successfully built.


--- Running Iterative Method ---
Starting Iterative Solver (Power Method)...
-> Converged in 86 iterations (0.0110 seconds).

--- Running Analytical Method ---
Starting Analytical Solver (Matrix Inversion)...
-> Analytical Matrix Inversion finished in 9.2747 seconds.

--- Top 5 Nodes (Iterative) ---
Node 486980: 0.00699902
Node 285814: 0.00474755
Node 226374: 0.00339558
Node 163075: 0.00333083
Node 555924: 0.00268606

--- Top 5 Nodes (Analytical) ---
Node 486980: 0.00699902
Node 285814: 0.00474755
Node 226374: 0.00339558
Node 163075: 0.00333083
Node 555924: 0.00268606

Total L1 Error between Iterative and Analytical vectors: 2.2144027658e-08
SUCCESS: The scalable iterative algorithm mathematically matches the closed-form derivat

# 3. Teleportation Probability Sensitivity
Analyze how changing teleportation probability `p` affects convergence speed and the PageRank score distribution.

In [3]:
# Sensitivity analysis for teleport probability p
p_values = [0.05, 0.15, 0.50]
sensitivity_rows = []

for p in p_values:
    engine_p = PageRankEngine(teleport_prob=p)
    engine_p.load_from_file(data_path)

    R_p = engine_p.solve_iterative(max_iter=150, tol=1e-8)
    top5 = engine_p.get_top_k(R_p, k=5)

    # Re-run once with loose tolerance to estimate convergence speed proxy
    it_engine = PageRankEngine(teleport_prob=p)
    it_engine.load_from_file(data_path)

    R_prev = np.ones(it_engine.N) / it_engine.N
    iter_count = 0
    for i in range(1, 151):
        link_distribution = (1 - it_engine.p) * it_engine.M.dot(R_prev)
        dead_end_mass = np.sum(R_prev[it_engine.dead_ends])
        total_teleport_mass = (it_engine.p + (1 - it_engine.p) * dead_end_mass) / it_engine.N
        R_new = link_distribution + total_teleport_mass

        if np.linalg.norm(R_new - R_prev, ord=1) <= 1e-8:
            iter_count = i
            break
        R_prev = R_new

    if iter_count == 0:
        iter_count = 150

    concentration_ratio = float(np.max(R_p) / np.min(R_p))
    sensitivity_rows.append((p, iter_count, concentration_ratio, top5))

print("Teleportation sensitivity summary:")
for p, iters, ratio, top5 in sensitivity_rows:
    top_nodes = [node for node, _ in top5]
    print(f"p={p:.2f} | iterations={iters:3d} | max/min={ratio:.2f} | top5={top_nodes}")

Loading graph from ../data/raw/web-Google_10k.zip...
Reading directly from web-Google_10k.txt inside the archive...
Extracting nodes and building index mappings...
Processing 10000 unique nodes and 78323 edges...
Sparse transition matrix M successfully built.

Starting Iterative Solver (Power Method)...
-> Warning: Reached max iterations (150) without strict convergence.
Loading graph from ../data/raw/web-Google_10k.zip...
Reading directly from web-Google_10k.txt inside the archive...
Extracting nodes and building index mappings...
Processing 10000 unique nodes and 78323 edges...
Sparse transition matrix M successfully built.

Loading graph from ../data/raw/web-Google_10k.zip...
Reading directly from web-Google_10k.txt inside the archive...
Extracting nodes and building index mappings...
Processing 10000 unique nodes and 78323 edges...
Sparse transition matrix M successfully built.

Starting Iterative Solver (Power Method)...
-> Converged in 86 iterations (0.0110 seconds).
Loading grap